# Week 7 — Multiple Regression as a Controlled Laboratory

Até agora:
- explorámos os dados
- entendemos variáveis
- analisámos relações

Hoje:
- vamos produzir diretrizes para a construção de um modelo de regressão linear múltipla

---

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import statsmodels.formula.api as smf
from statsmodels.stats.outliers_influence import variance_inflation_factor


## Dataset: California Housing

Vamos usar este dataset como laboratório.

Target:
- MedHouseVal (valor mediano das casas)

---

In [2]:
#leitura do dataset usando pandas
data_path = './data/raw/housing.csv'

df = pd.read_csv(data_path)
df.head()

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity
0,-122.23,37.88,41.0,880.0,129.0,322.0,126.0,8.3252,452600.0,NEAR BAY
1,-122.22,37.86,21.0,7099.0,1106.0,2401.0,1138.0,8.3014,358500.0,NEAR BAY
2,-122.24,37.85,52.0,1467.0,190.0,496.0,177.0,7.2574,352100.0,NEAR BAY
3,-122.25,37.85,52.0,1274.0,235.0,558.0,219.0,5.6431,341300.0,NEAR BAY
4,-122.25,37.85,52.0,1627.0,280.0,565.0,259.0,3.8462,342200.0,NEAR BAY


## Variáveis

- medianHouseValue → target
- medianIncome → rendimento
- totalRooms / totalBedrooms → estrutura da casa
- population / households → densidade
- oceanProximity → localização categórica

## Preparação

Este dataset tem algo importante:

- missing values reais

Particularmente em:
- total_bedrooms

---

In [3]:
#check missing values in totalBedrooms
missing_bedrooms = df['total_bedrooms'].isnull().sum()
print(f"Missing values in totalBedrooms: {missing_bedrooms}")

Missing values in totalBedrooms: 207


In [4]:
df.isnull().sum()

longitude               0
latitude                0
housing_median_age      0
total_rooms             0
total_bedrooms        207
population              0
households              0
median_income           0
median_house_value      0
ocean_proximity         0
dtype: int64

In [5]:
# decisão explícita (não automática!)
df_model = df.copy()

# opção simples (para aula)
df_model["total_bedrooms"] = df_model["total_bedrooms"].fillna(
    df_model["total_bedrooms"].median()
)

## Feature engineering

Vamos criar variáveis mais interpretáveis:

- rooms_per_household
- bedrooms_per_room
- population_per_household

In [6]:
#estrutura da casa
df_model["bedrooms_per_room"] = df_model["total_bedrooms"] / df_model["total_rooms"]
#densidade
df_model["population_per_household"] = df_model["population"] / df_model["households"]
#criar log de house value
df_model["log_median_house_value"] = np.log(df_model["median_house_value"])
#isso é feito para facilitar a interpretação dos coeficientes do modelo de regressão linear múltipla

In [7]:
#convert ocean_proximity usando one hot encodding
ocean_hot_encoded = pd.get_dummies(df_model["ocean_proximity"], drop_first=True)
df_model = pd.concat([df_model, ocean_hot_encoded], axis=1)

In [8]:
df_model["ocean_proximity"].value_counts()

ocean_proximity
<1H OCEAN     9136
INLAND        6551
NEAR OCEAN    2658
NEAR BAY      2290
ISLAND           5
Name: count, dtype: int64

In [9]:
#split df_model em treino e teste
train_df, test_df = train_test_split(df_model, test_size=0.3, random_state=42)

In [10]:
#fit model considerando as variáveis numéricas
model  = smf.ols('log_median_house_value ~ median_income + housing_median_age + bedrooms_per_room + population_per_household', data=train_df).fit()

#print summary
print(model.summary())

                              OLS Regression Results                              
Dep. Variable:     log_median_house_value   R-squared:                       0.501
Model:                                OLS   Adj. R-squared:                  0.501
Method:                     Least Squares   F-statistic:                     3620.
Date:                    Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                            17:42:49   Log-Likelihood:                -7339.5
No. Observations:                   14448   AIC:                         1.469e+04
Df Residuals:                       14443   BIC:                         1.473e+04
Df Model:                               4                                         
Covariance Type:                nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------------

In [11]:
#test model on test set
test_df["predicted_log_median_house_value"] = model.predict(test_df)

#calcular RMSE
rmse = np.sqrt(mean_squared_error(test_df["log_median_house_value"], test_df["predicted_log_median_house_value"]))
print(f"RMSE: {rmse}")

RMSE: 0.43507252645299727


In [12]:
#formula de RMSE:
# RMSE = sqrt(mean((y_true - y_pred)^2))
#calcular from scratch
a = (sum((test_df["log_median_house_value"] - test_df["predicted_log_median_house_value"])**2)/len(test_df))**0.5
print(a)


0.43507252645299727


In [13]:
#como interepretar rmse:
#- RMSE é a raiz quadrada da média dos erros quadráticos
#- quanto menor o RMSE, melhor o modelo se encaixa aos dados
#- RMSE é sensível a outliers, pois os erros quadráticos amplificam os erros grandes

#comparar sempre o RMSE com a unidade de analise e com a média


In [14]:
#fit model considerando as variáveis numéricas
model2  = smf.ols('median_house_value ~ median_income + housing_median_age + bedrooms_per_room + population_per_household + Q("NEAR OCEAN") ', data=train_df).fit()

#print summary
print(model2.summary())

test_df["predicted_model2"] = model2.predict(test_df)

                            OLS Regression Results                            
Dep. Variable:     median_house_value   R-squared:                       0.561
Model:                            OLS   Adj. R-squared:                  0.561
Method:                 Least Squares   F-statistic:                     3697.
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:42:49   Log-Likelihood:            -1.8300e+05
No. Observations:               14448   AIC:                         3.660e+05
Df Residuals:                   14442   BIC:                         3.661e+05
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [15]:
#calcular RMSE
rmse = np.sqrt(mean_squared_error(test_df["median_house_value"], test_df["predicted_model2"]))
print(f"RMSE: {rmse}")

RMSE: 82013.08512135045


In [16]:
#fit model considerando as variáveis numéricas
model3  = smf.ols('median_house_value ~ median_income + housing_median_age + bedrooms_per_room + population_per_household + Q("NEAR OCEAN") + Q("NEAR BAY") + ISLAND + INLAND ', data=train_df).fit()

#print summary
print(model3.summary())

test_df["predicted_model3"] = model3.predict(test_df)

                            OLS Regression Results                            
Dep. Variable:     median_house_value   R-squared:                       0.611
Model:                            OLS   Adj. R-squared:                  0.611
Method:                 Least Squares   F-statistic:                     2838.
Date:                Mon, 13 Apr 2026   Prob (F-statistic):               0.00
Time:                        17:42:49   Log-Likelihood:            -1.8213e+05
No. Observations:               14448   AIC:                         3.643e+05
Df Residuals:                   14439   BIC:                         3.643e+05
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
                               coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------------------
Intercept               

In [17]:
#calcular RMSE
rmse = np.sqrt(mean_squared_error(test_df["median_house_value"], test_df["predicted_model3"]))
print(f"RMSE: {rmse}")

RMSE: 74389.51342355102


In [18]:
#sobre a leitura do R2 e o R2 ajustado (penaliza o modelo por adicionar variáveis que não contribuem para a explicação da variância)
#- R2 é a proporção da variância explicada pelo modelo
#- R2 ajustado é o R2 penalizado pelo número de variáveis no modelo

In [19]:
#calular o VIF para cada variável
# VIF = 1 / (1 - R2_i), onde R2_i é o R2 do modelo que regrediu a variável i contra as outras variáveis
X = train_df[["median_income", "housing_median_age", "bedrooms_per_room", "population_per_household"]]
vif_data = pd.DataFrame()
# vif_data
vif_data["feature"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

                    feature       VIF
0             median_income  3.045535
1        housing_median_age  5.713644
2         bedrooms_per_room  5.582586
3  population_per_household  1.065169


In [20]:
#tabela para avaliar VIF
# VIF > 5 indica multicolinearidade moderada
# VIF > 10 indica multicolinearidade severa


In [21]:
df_model["price_tier"] = pd.qcut(
    df_model["median_house_value"],
    q=3,
    labels=["Low", "Medium", "High"]
)

In [25]:
#one hot encoding price_tier
price_hot = pd.get_dummies(df_model["price_tier"], drop_first=True)
df_model = pd.concat([df_model, price_hot], axis=1)

In [26]:
#split df_model em treino e teste
train_df, test_df = train_test_split(df_model, test_size=0.3, random_state=42)


In [27]:
train_df

,longitude,latitude,housing_median_age,total_rooms,total_bedrooms,population,households,median_income,median_house_value,ocean_proximity,bedrooms_per_room,population_per_household,log_median_house_value,INLAND,ISLAND,NEAR BAY,NEAR OCEAN,price_tier,Medium,High
7061,-118.02,33.93,35.0,2400.0,398.0,1218.0,408.0,4.1312,193800.0,<1H OCEAN,0.165833,2.985294,12.174582,False,False,False,False,Medium,True,False
14689,-117.09,32.79,20.0,2183.0,534.0,999.0,496.0,2.8631,169700.0,NEAR OCEAN,0.244617,2.014113,12.041787,False,False,False,True,Medium,True,False
17323,-120.14,34.59,24.0,1601.0,282.0,731.0,285.0,4.2026,259800.0,NEAR OCEAN,0.176140,2.564912,12.467667,False,False,False,True,High,False,True
10056,-121.00,39.26,14.0,810.0,151.0,302.0,138.0,3.1094,136100.0,INLAND,0.186420,2.188406,11.821145,True,False,False,False,Low,False,False
15750,-122.45,37.77,52.0,3188.0,708.0,1526.0,664.0,3.3068,500001.0,NEAR BAY,0.222083,2.298193,13.122365,False,False,True,False,High,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11284,-117.96,33.78,35.0,1330.0,201.0,658.0,217.0,6.3700,229200.0,<1H OCEAN,0.151128,3.032258,12.342350,False,False,False,False,Medium,True,False
11964,-117.43,34.02,33.0,3084.0,570.0,1753.0,449.0,3.0500,97800.0,INLAND,0.184825,3.904232,11.490680,True,False,False,False,Low,False,False
5390,-118.38,34.03,36.0,2101.0,569.0,1756.0,527.0,2.9344,222100.0,<1H OCEAN,0.270823,3.332068,12.310883,False,False,False,False,Medium,True,False
860,-121.96,37.58,15.0,3575.0,597.0,1777.0,559.0,5.7192,283500.0,<1H OCEAN,0.166993,3.178891,12.554967,False,False,False,False,High,False,True


In [31]:
#usar formula para criar um modelo de regressao logistica
train_df["High"] = train_df["High"].astype(int)
modelhight  = smf.logit('High ~ median_income + housing_median_age + bedrooms_per_room + population_per_household + Q("NEAR OCEAN")', data=train_df).fit(maxiter=200)

Optimization terminated successfully.
         Current function value: 0.391089
         Iterations 8


In [33]:
#predizer probabilidades de ser High na base de teste
test_df["predicted_prob_high"] = modelhight.predict(test_df)

#matriz de confusão
test_df["predicted_high"] = (test_df["predicted_prob_high"] > 0.5).astype(int)
confusion_matrix = pd.crosstab(test_df["High"], test_df["predicted_high"])
print(confusion_matrix)

predicted_high     0     1
High                      
False           3759   351
True             711  1371


In [36]:
#calculo do f1 score
from sklearn.metrics import f1_score, accuracy_score

f1 = f1_score(test_df["High"], test_df["predicted_high"])
print(f"F1 Score: {f1}")

accuracy = accuracy_score(test_df["High"], test_df["predicted_high"])
print(f"Accuracy Score: {accuracy}")



F1 Score: 0.7208201892744479
Accuracy Score: 0.8284883720930233
